In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install necessary libraries
!pip install datasets transformers --quiet

# Import libraries
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from huggingface_hub import login

# Login to Hugging Face Hub using your token
login(token="your token")

# Load the dataset
dataset = load_dataset("sapienzanlp/nlp2025_hw1_cultural_dataset")

# Display the first few rows of the training set
df_train = pd.DataFrame(dataset["train"])
df_train.head()


In [ ]:
# Import necessary classes from the transformers library
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch


In [ ]:
# Define a preprocessing function to combine relevant fields into a single text field
def preprocess(example):
    example["text"] = example["name"] + " - " + example["description"] + " - " + example["type"]  + " - " + example["category"]
    return example


In [ ]:
# Load the tokenizer for the DistilBERT model
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Define a tokenization function to tokenize the combined text field
def tokenize(example):
    return tokenizer(example["text"], truncation=True)


## TRAINING

In [ ]:
# Plot the distribution of classes in the training set
sns.countplot(x="label", data=df_train)
plt.title("Distribution of Classes in the Training Set")
plt.xlabel("Classification")
plt.ylabel("Number of Items")
plt.show()


In [ ]:
# Apply the preprocessing function to the training and validation sets
dataset["train"] = dataset["train"].map(preprocess)
dataset["validation"] = dataset["validation"].map(preprocess)


In [ ]:
# Apply the tokenization function to the training and validation sets
tokenized_train = dataset["train"].map(tokenize, batched=True)
tokenized_dev = dataset["validation"].map(tokenize, batched=True)


In [ ]:
# Create dictionaries to map labels to IDs and vice versa
label2id = {"cultural agnostic": 0, "cultural representative": 1, "cultural exclusive": 2}
id2label = {v: k for k, v in label2id.items()}

# Define a function to encode labels into numerical IDs
def encode_labels(example):
    example["label"] = label2id[example["label"].lower()]
    return example

# Apply the label encoding function to the tokenized training and validation sets
tokenized_train = tokenized_train.map(encode_labels)
tokenized_dev = tokenized_dev.map(encode_labels)


In [ ]:
# Load the DistilBERT model for sequence classification
distil_bert = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,  # Number of labels for classification
    id2label=id2label,  # Mapping from IDs to labels
    label2id=label2id   # Mapping from labels to IDs
)


In [ ]:
# Freeze the parameters of the first three layers of the DistilBERT model
for param in distil_bert.distilbert.transformer.layer[:3]:
    for p in param.parameters():
        p.requires_grad = False  # Set requires_grad to False to prevent these parameters from being updated during training


In [ ]:
# Load the roBERTa model for sequence classification
bert = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3,  # Number of labels for classification
    id2label=id2label,  # Mapping from IDs to labels
    label2id=label2id   # Mapping from labels to IDs
)

In [ ]:
# Freeze the parameters of the first three layers of the roBERTa model
for param in bert.roberta.encoder.layer[:3]: # Access encoder.layer instead of transformer.layer
    for p in param.parameters():
        p.requires_grad = False  # Set requires_grad to False to prevent these parameters from being updated during training

In [ ]:
from transformers import TrainerCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import json
import os

# Define a function to compute evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Custom callback to calculate and print metrics every few steps
class MetricsCallback(TrainerCallback):
    def __init__(self, logging_steps=10):
        self.logging_steps = logging_steps

    def on_step_end(self, args, state, control, **kwargs):
        if "outputs" in kwargs and state.global_step % self.logging_steps == 0:
            # Calculate metrics
            predictions, labels = kwargs['outputs']
            metrics = compute_metrics((predictions, labels))
            print(f"Step {state.global_step}: {metrics}")
        return control

# Custom callback to log metrics to a JSON file
class JSONLoggerCallback(TrainerCallback):
    def __init__(self, log_file_path="./logs/training_logs.jsonl"):
        self.log_file_path = log_file_path
        os.makedirs(os.path.dirname(log_file_path), exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            with open(self.log_file_path, "a") as f:
                logs["step"] = state.global_step
                json.dump(logs, f)
                f.write("\n")


In [ ]:
# Set the device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",  # Directory to save the model checkpoints
    eval_strategy="steps",  # Evaluation strategy to use
    save_strategy="epoch",  # Save the model at the end of each epoch
    per_device_train_batch_size=64,  # Batch size for training
    per_device_eval_batch_size=64,  # Batch size for evaluation
    num_train_epochs=3,  # Number of training epochs
    weight_decay=0.01,  # Weight decay for regularization
    logging_steps=10,  # Log metrics every 10 steps
    logging_dir="./logs",  # Directory to save the logs
    report_to="tensorboard"  # Report metrics to TensorBoard
)

In [ ]:
# Move the model to the GPU if available
distil_bert.to(device)

# Initialize the Trainer
trainer_distilbert = Trainer(
    model=distil_bert,  # The model to be trained
    args=training_args,  # Training arguments
    train_dataset=tokenized_train,  # Training dataset
    eval_dataset=tokenized_dev,  # Evaluation dataset
    tokenizer=tokenizer,  # Tokenizer
    data_collator=DataCollatorWithPadding(tokenizer),  # Data collator for padding
    compute_metrics=compute_metrics,  # Function to compute metrics
    callbacks=[JSONLoggerCallback()]  # Custom callback for logging
)

# Start training
trainer_distilbert.train()


In [ ]:
# Move the model to the GPU if available
bert.to(device)

# Initialize the Trainer
trainer_bert = Trainer(
    model=bert,  # The model to be trained
    args=training_args,  # Training arguments
    train_dataset=tokenized_train,  # Training dataset
    eval_dataset=tokenized_dev,  # Evaluation dataset
    tokenizer=tokenizer,  # Tokenizer
    data_collator=DataCollatorWithPadding(tokenizer),  # Data collator for padding
    compute_metrics=compute_metrics,  # Function to compute metrics
    callbacks=[JSONLoggerCallback()]  # Custom callback for logging
)

# Start training
trainer_bert.train()


In [ ]:
# Choose the model (we choose distil_bert beacuse have the best accuracy)
trainer = trainer_distilbert

In [ ]:
# Save the final model at the end of training
trainer.save_model("/content/drive/MyDrive/The_Giadas_shared_folder/final_model1")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay # Import confusion_matrix and ConfusionMatrixDisplay
import numpy as np
# Read logs from the file
log_path = "./logs/training_logs.jsonl"
log_data = []

with open(log_path, 'r') as f:
    for line in f:
        try:
            log_data.append(json.loads(line))
        except:
            pass

log_df = pd.DataFrame(log_data)

# Plot the loss during training
plt.figure(figsize=(10, 6))
sns.lineplot(data=log_df, x="step", y="loss")
plt.title("Loss During Training")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# Plot the accuracy during training
plt.figure(figsize=(10, 6))
sns.lineplot(data=log_df, x="step", y="eval_accuracy")
plt.title("Accuracy During Training")
plt.xlabel("Step")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

# Plot the precision during training
plt.figure(figsize=(10, 6))
sns.lineplot(data=log_df, x="step", y="eval_precision")
plt.title("Precision During Training")
plt.xlabel("Step")
plt.ylabel("Precision")
plt.grid(True)
plt.show()

# Plot the recall during training
plt.figure(figsize=(10, 6))
sns.lineplot(data=log_df, x="step", y="eval_recall")
plt.title("Recall During Training")
plt.xlabel("Step")
plt.ylabel("Recall")
plt.grid(True)
plt.show()

# Plot the F1 score during training
plt.figure(figsize=(10, 6))
sns.lineplot(data=log_df, x="step", y="eval_f1")
plt.title("F1 Score During Training")
plt.xlabel("Step")
plt.ylabel("F1")
plt.grid(True)
plt.show()

# Valutazione sul set di validazione
predictions = trainer.predict(tokenized_dev)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids
# Calcola la confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Opzionale: se hai nomi delle classi
labels = ["cultural agnostic", "cultural representative", "cultural exclusive"]

# Visualizza la confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')  # Colormap opzionale
plt.title("Confusion Matrix")
plt.show()

## TEST

In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import Trainer
import os
os.environ["WANDB_DISABLED"] = "true"
from transformers import AutoTokenizer, TrainingArguments # Import TrainingArguments

# Carica il modello salvato
model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/The_Giadas_shared_folder/final_model1")

# Carica il tokenizer salvato insieme al modello
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/The_Giadas_shared_folder/final_model1")


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    save_strategy="epoch",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    logging_dir="./logs",
    report_to="flag"
)


trainer = Trainer(
    model=model,
    processing_class=tokenizer
)

In [ ]:
# Step 1: Prepara il test set
test_dataset = pd.read_csv('/content/drive/MyDrive/The_Giadas_shared_folder/test_unlabeled.csv')

# Convert the DataFrame to a list of dictionaries
test_dataset = test_dataset.to_dict('records')

# Apply the preprocessing function to each dictionary in the list
processed_data = [preprocess(example) for example in test_dataset]

# Convert the processed list of dictionaries back to a DataFrame
processed_df = pd.DataFrame(processed_data)

# Convert the pandas DataFrame to a Hugging Face Dataset
import datasets
test_dataset_hf = datasets.Dataset.from_pandas(processed_df)

# Se hai già queste funzioni definite prima (preprocess e tokenize), le riutilizzi:
# Use the Hugging Face Dataset here
tokenized_test = test_dataset_hf.map(tokenize, batched=True)


# Step 2: Genera predizioni
pred_output = trainer.predict(tokenized_test)
pred_classes = pred_output.predictions.argmax(axis=1)

# Mappa i numeri in etichette testuali
id2label = {
    0: "cultural agnostic",
    1: "cultural representative",
    2: "cultural exclusive"
}
pred_labels = [id2label[i] for i in pred_classes]

# Step 3: Salva in CSV
import pandas as pd

# Recupera le colonne originali dal test set
# Use the processed DataFrame here as well
df_test = pd.DataFrame(processed_df)
df_test["label"] = pred_labels

# Salva
df_test.to_csv("/content/drive/MyDrive/The_Giadas_shared_folder/The_Giadas_output_modello1.csv", index=False)


In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.countplot(x=pred_labels, order=sorted(set(pred_labels)))
plt.title("Distribution of the Predictions per Classes")
plt.xlabel("Predicted classes")
plt.ylabel("Number of instances")
plt.xticks(rotation=45)
plt.show()
